In [1]:
# ============================================================
# TRANSFER LEARNING CON VGG16
# ============================================================
#
# Escenario:
#   Tenemos una CNN entrenada en ImageNet
#   y queremos adaptarla a un problema NUEVO.
#
# Dataset elegido:
#   Oxford-IIIT Pet Dataset
#
# ¿Por qué este dataset?
# - perros y gatos
# - MUY distinto a ImageNet clásico
# - visualmente interesante
# - suficientemente pequeño para entrenar en vivo
#
# Vamos a comparar:
#
# 1) Entrenar SOLO la última capa
# 2) Fine-tuning completo
#
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cuda


In [2]:
# ============================================================
# 1. DATASET
# ============================================================

# VGG16 espera:
# - imágenes 224x224
# - normalización ImageNet

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [3]:

# ============================================================
# Oxford-IIIT Pet Dataset
# ============================================================

full_dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=train_transform
)

# El dataset tiene 37 clases

num_classes = 37

# Split train/test rápido para demo

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size]
)

# Cambiamos transform del test
test_dataset.dataset.transform = test_transform

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Train samples:", len(train_dataset))
print("Test samples :", len(test_dataset))

Train samples: 2944
Test samples : 736


In [4]:
# ============================================================
# 2. CARGAR VGG16 PREENTRENADA
# ============================================================

model = models.vgg16(weights="IMAGENET1K_V1")

print(model.features)
print(model.classifier)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (17): Conv2d(256, 512, kernel_si

In [5]:

# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

criterion = nn.CrossEntropyLoss()

def count_trainable_params(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

def create_model(num_classes):

    model = models.vgg16(weights="IMAGENET1K_V1") # <--- weights = None
    
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, num_classes)

    return model

def train_one_epoch(model, loader, optimizer):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total

    return avg_loss, acc


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total

    return avg_loss, acc


In [6]:
# ============================================================
# 5. FEATURE EXTRACTION
# ============================================================
#
# Entrenamos SOLO la última capa.
#
# La CNN funciona como extractor de features.
#
# ============================================================

model = create_model(num_classes).to(device)

print("\n================================================")
print("FEATURE EXTRACTION")
print("Solo entrenamos la última capa")
print("================================================")

# congelamos extractor visual

for param in model.parameters():
    param.requires_grad = False

# entrenamos la última capa del classifier

for param in model.classifier[6].parameters():
    param.requires_grad = True

# Mostramos parámetros entrenables

print("Parámetros Entrenables:", count_trainable_params(model))

optimizer = optim.Adam(model.classifier.parameters(),lr=1e-3)
epochs = 5

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.2f}%")

    print(f"Test Loss : {test_loss:.4f}")
    print(f"Test Acc  : {test_acc:.2f}%")

feature_extraction_acc = test_acc


FEATURE EXTRACTION
Solo entrenamos la última capa
Parámetros Entrenables: 151589

Epoch 1/5


100%|██████████| 92/92 [00:05<00:00, 18.23it/s]


Train Loss: 0.9122
Train Acc : 73.64%
Test Loss : 0.4193
Test Acc  : 86.68%

Epoch 2/5


100%|██████████| 92/92 [00:04<00:00, 18.83it/s]


Train Loss: 0.3711
Train Acc : 87.81%
Test Loss : 0.3612
Test Acc  : 87.77%

Epoch 3/5


...

Train Loss: 0.2943
Train Acc : 89.71%
Test Loss : 0.3629
Test Acc  : 87.23%

Epoch 4/5


100%|██████████| 92/92 [00:04<00:00, 18.68it/s]


Train Loss: 0.2529
Train Acc : 91.58%
Test Loss : 0.3260
Test Acc  : 88.99%

Epoch 5/5


100%|██████████| 92/92 [00:04<00:00, 18.75it/s]


Train Loss: 0.2188
Train Acc : 92.63%
Test Loss : 0.3169
Test Acc  : 88.86%


In [7]:

# ============================================================
# 6. FULL FINE-TUNING
# ============================================================
#
# Ahora entrenamos TODA la red.
#
# ============================================================

print("\n================================================")
print("FULL FINE-TUNING")
print("Entrenamos toda la red")
print("================================================")

model = create_model(num_classes).to(device)

# Descongelamos TODO

for param in model.parameters():
    param.requires_grad = True

print("Parámetros Entrenables:", count_trainable_params(model))

optimizer = optim.Adam(model.parameters(),lr=1e-4)
epochs = 5

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.2f}%")

    print(f"Test Loss : {test_loss:.4f}")
    print(f"Test Acc  : {test_acc:.2f}%")

full_finetune_acc = test_acc



FULL FINE-TUNING
Entrenamos toda la red
Parámetros Entrenables: 134412133

Epoch 1/5


100%|██████████| 92/92 [00:16<00:00,  5.69it/s]


Train Loss: 1.5427
Train Acc : 55.37%
Test Loss : 0.5756
Test Acc  : 82.07%

Epoch 2/5


100%|██████████| 92/92 [00:16<00:00,  5.63it/s]


Train Loss: 0.5753
Train Acc : 82.44%
Test Loss : 0.6010
Test Acc  : 80.03%

Epoch 3/5


...

Train Loss: 0.3822
Train Acc : 87.84%
Test Loss : 0.4696
Test Acc  : 85.46%

Epoch 4/5


100%|██████████| 92/92 [00:16<00:00,  5.58it/s]


Train Loss: 0.2802
Train Acc : 90.69%
Test Loss : 0.5125
Test Acc  : 84.51%

Epoch 5/5


100%|██████████| 92/92 [00:16<00:00,  5.54it/s]


Train Loss: 0.2122
Train Acc : 93.24%
Test Loss : 0.4782
Test Acc  : 85.73%


In [ ]:

# ============================================================
# 7. COMPARACIÓN FINAL
# ============================================================

print("\n================================================")
print("RESULTADOS")
print("================================================")

print(f"Solo última capa : {feature_extraction_acc:.2f}%")
print(f"Fine-tuning total: {full_finetune_acc:.2f}%")

"""
============================================================
¿QUÉ ESTÁ PASANDO?
============================================================

La VGG16 ya fue entrenada en millones de imágenes
(ImageNet).

Eso significa que YA aprendió:
- bordes
- texturas
- formas
- patrones visuales generales

Aunque nunca haya visto exactamente estas razas
de perros y gatos.

============================================================
FEATURE EXTRACTION
============================================================

Congelamos toda la CNN.

La usamos solamente como:

    extractor de features

Solo aprende:
- la última capa lineal

Ventajas:
- muy rápido
- poco riesgo de overfitting
- ideal con datasets chicos

============================================================
FULL FINE-TUNING
============================================================

Entrenamos TODA la red.

Ahora:
- TODAS las capas se adaptan
- los features cambian
- la red se especializa

Ventajas:
- generalmente mejor accuracy

Desventajas:
- más lento
- más memoria
- puede overfittear

============================================================
¿POR QUÉ LR MÁS CHICO?
============================================================

Los pesos preentrenados ya son buenos.

No queremos destruirlos con updates gigantes.

Por eso:
- feature extraction -> LR más grande
- fine-tuning total -> LR más chico

"""


RESULTADOS
Solo última capa : 88.86%
Fine-tuning total: 85.73%


'\n============================================================\n¿QUÉ ESTÁ PASANDO?\n============================================================\n\nLa ResNet18 ya fue entrenada en millones de imágenes\n(ImageNet).\n\nEso significa que YA aprendió:\n- bordes\n- texturas\n- formas\n- patrones visuales generales\n\nAunque nunca haya visto exactamente estas razas\nde perros y gatos.\n\n============================================================\nFEATURE EXTRACTION\n============================================================\n\nCongelamos toda la CNN.\n\nLa usamos solamente como:\n\n    extractor de features\n\nSolo aprende:\n- la última capa lineal\n\nVentajas:\n- muy rápido\n- poco riesgo de overfitting\n- ideal con datasets chicos\n\n============================================================\nFULL FINE-TUNING\n============================================================\n\nEntrenamos TODA la red.\n\nAhora:\n- TODAS las capas se adaptan\n- los features cambian\n- la red se especializ